# ⚒️ TitleForge AI v2 — Full Training Notebook
**Run ALL cells top to bottom. At the end, the GGUF file will auto-download to your machine.**

> ⚠️ Set Runtime → T4 GPU before running!

| Step | Description |
|------|-------------|
| 1 | GPU check |
| 2 | Install deps |
| 3 | Load dataset |
| 4 | Load model (4-bit) |
| 5 | LoRA config |
| 6 | Train |
| 7 | Merge & save |
| 8 | Evaluate |
| 9 | Export GGUF → download |

In [ ]:
# Step 1 — GPU Check
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
assert torch.cuda.is_available(), '❌ No GPU! Go to Runtime → Change Runtime Type → T4 GPU'
dev = torch.cuda.get_device_properties(0)
print(f'✅ GPU: {dev.name}  ({dev.total_memory/1e9:.1f} GB VRAM)')
print(f'   BF16: {torch.cuda.is_bf16_supported()}')

In [ ]:
%%capture
# Step 2 — Install Dependencies
!pip install -q transformers==4.41.0 datasets peft trl bitsandbytes accelerate
!pip install -q sacrebleu rouge-score jiwer scikit-learn
print('✅ Packages installed')

In [ ]:
# Step 3 — Load Dataset
# Paste the raw content of sample_dataset.csv below (or upload via files.upload)
import io, pandas as pd
from sklearn.model_selection import train_test_split
from datasets import Dataset

# AUTO-LOAD: upload the file
from google.colab import files
print('📂 Upload sample_dataset.csv from f:\\Gen-Ai\\TitleForge-AI\\data\\')
uploaded = files.upload()
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname])).dropna(subset=['raw_title','clean_title'])
print(f'✅ Loaded {len(df)} pairs')

PROMPT = (
    '### Instruction:\n'
    'Normalize this raw product title into a clean, standardized catalog title. '
    'Fix capitalization, expand abbreviations, remove noise, and ensure consistent formatting.\n\n'
    '### Input:\n{raw}\n\n'
    '### Response:\n{clean}'
)
df['text'] = df.apply(lambda r: PROMPT.format(raw=r.raw_title.strip(), clean=r.clean_title.strip()), axis=1)

train_df, tmp = train_test_split(df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(tmp, test_size=0.5, random_state=42)
train_ds = Dataset.from_pandas(train_df[['text']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['text']].reset_index(drop=True))
print(f'Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}')
print('\n--- Sample ---')
print(train_df.iloc[0]['text'])

In [ ]:
# Step 4 — Load Model with 4-bit QLoRA
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
USE_BF16 = torch.cuda.is_bf16_supported()

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16 if USE_BF16 else torch.float16,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', trust_remote_code=True
)
model = prepare_model_for_kbit_training(model)
print('✅ TinyLlama loaded with 4-bit NF4')

In [ ]:
# Step 5 — LoRA Config
from peft import LoraConfig, get_peft_model, TaskType

lora = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','v_proj','k_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
# Step 6 — Fine-Tune with SFTTrainer
from trl import SFTTrainer, SFTConfig

ADAPTER = '/content/titleforge-adapter'
USE_BF16 = torch.cuda.is_bf16_supported()

args = SFTConfig(
    output_dir=ADAPTER,
    num_train_epochs=5,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=not USE_BF16, bf16=USE_BF16,
    logging_steps=5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    report_to='none',
    max_seq_length=256,
    dataset_text_field='text',
    optim='paged_adamw_32bit',
)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=train_ds, eval_dataset=val_ds, args=args,
)
print('🏋️  Training started (~10-15 min on T4)...')
trainer.train()
print('✅ Training complete!')

In [ ]:
# Step 7a — Save adapter
trainer.save_model(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print(f'✅ Adapter saved → {ADAPTER}')

In [ ]:
# Step 7b — Merge adapter into full model
from peft import PeftModel
MERGED = '/content/titleforge-merged'

print('🔀 Merging LoRA → base model (runs on CPU, ~2 min)...')
base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='cpu', trust_remote_code=True
)
merged = PeftModel.from_pretrained(base, ADAPTER)
merged = merged.merge_and_unload()
merged.save_pretrained(MERGED, safe_serialization=True)
tokenizer.save_pretrained(MERGED)
print(f'✅ Merged model saved → {MERGED}')

In [ ]:
# Step 8 — Evaluation
from transformers import pipeline as hf_pipeline
from rouge_score import rouge_scorer as rs
import sacrebleu
from jiwer import cer as compute_cer

INFER_TMPL = (
    '### Instruction:\nNormalize this raw product title into a clean, standardized catalog title. '
    'Fix capitalization, expand abbreviations, remove noise, and ensure consistent formatting.\n\n'
    '### Input:\n{raw}\n\n### Response:\n'
)
pipe = hf_pipeline(
    'text-generation', model=MERGED,
    torch_dtype=torch.float16, device_map='auto',
    max_new_tokens=80, do_sample=False, repetition_penalty=1.1
)
def infer(raw):
    out = pipe(INFER_TMPL.format(raw=raw))[0]['generated_text']
    return out.split('### Response:\n')[-1].split('\n')[0].strip()

preds, refs = [], []
scorer = rs.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)
for _, row in test_df.iterrows():
    preds.append(infer(row['raw_title']))
    refs.append(row['clean_title'])

r1 = sum(scorer.score(r,p)['rouge1'].fmeasure for p,r in zip(preds,refs))/len(preds)
r2 = sum(scorer.score(r,p)['rouge2'].fmeasure for p,r in zip(preds,refs))/len(preds)
rl = sum(scorer.score(r,p)['rougeL'].fmeasure for p,r in zip(preds,refs))/len(preds)
bleu = sum(sacrebleu.sentence_bleu(p,[r]).score for p,r in zip(preds,refs))/len(preds)
em = sum(p.strip().lower()==r.strip().lower() for p,r in zip(preds,refs))/len(preds)*100
cer_score = sum(compute_cer(r,p) for p,r in zip(preds,refs))/len(preds)

print('='*50)
print('📊 EVALUATION RESULTS')
print('='*50)
print(f'  BLEU          : {bleu:.2f}')
print(f'  ROUGE-1       : {r1:.4f}')
print(f'  ROUGE-2       : {r2:.4f}')
print(f'  ROUGE-L       : {rl:.4f}')
print(f'  Exact Match   : {em:.1f}%')
print(f'  Char Error Rate: {cer_score:.4f}')
print('\n📋 Sample Predictions:')
for i in range(min(5,len(preds))):
    print(f'  RAW : {test_df.iloc[i]["raw_title"]}')
    print(f'  PRED: {preds[i]}')
    print(f'  GOLD: {refs[i]}\n')

In [ ]:
%%bash
# Step 9a — Clone llama.cpp and install Python deps
git clone --depth 1 https://github.com/ggerganov/llama.cpp /content/llama.cpp
pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
echo '✅ llama.cpp ready'

In [ ]:
%%bash
# Step 9b — Convert to F16 GGUF
python /content/llama.cpp/convert_hf_to_gguf.py \
    /content/titleforge-merged \
    --outtype f16 \
    --outfile /content/titleforge-f16.gguf
echo '✅ F16 GGUF created:'
ls -lh /content/titleforge-f16.gguf

In [ ]:
%%bash
# Step 9c — Build llama-quantize and quantize to Q4_K_M
cd /content/llama.cpp
make -j$(nproc) llama-quantize 2>&1 | tail -3
./llama-quantize /content/titleforge-f16.gguf /content/titleforge-q4_K_M.gguf q4_K_M
echo '✅ Q4_K_M GGUF created:'
ls -lh /content/titleforge-q4_K_M.gguf

In [ ]:
# Step 9d — Download the GGUF to your machine
from google.colab import files
print('📥 Downloading titleforge-q4_K_M.gguf ...')
files.download('/content/titleforge-q4_K_M.gguf')
print('✅ Download started!')
print()
print('='*60)
print('NEXT STEPS on your Windows machine:')
print('='*60)
print('1. Move the .gguf file to:')
print('   f:\\Gen-Ai\\TitleForge-AI\\export\\gguf\\')
print()
print('2. Run this PowerShell script:')
print('   f:\\Gen-Ai\\TitleForge-AI\\import_to_ollama.ps1')
print()
print('3. Start the API:')
print('   cd f:\\Gen-Ai\\TitleForge-AI')
print('   $env:MODEL_BACKEND="ollama"; uvicorn app.main:app --reload')

In [ ]:
# Step 10 — Quick Inference Demo (in Colab)
test_titles = [
    'APPLE IPHONE 15 PRO MAX 256GB BLK',
    'sony wh1000xm5 hdphn blk wrls noisecancelling',
    '!!SALE!! Nike Air Max 90 sz10 wht',
    'Memory Card 64gb class 10 sd samsung',
    'DELL INSPIRON 15 LAPTOP I7 16GB RAM 512SSD WIN11',
    'lg 55inch oled tv 4k smart webos 2023',
    'bosch cordless drill 18v 2x2.0ah battry set',
]
print('🔍 TitleForge AI — Fine-tuned Inference Demo')
print('='*60)
for raw in test_titles:
    out = infer(raw)
    print(f'  IN : {raw}')
    print(f'  OUT: {out}\n')